# Infino

>[Infino](https://github.com/infinohq/infino) 是一个可扩展的遥测存储库，专为日志、指标和追踪而设计。Infino 可以作为独立的观测解决方案运行，也可以作为您观测栈中的存储层。

本示例展示了如何通过 `LangChain` 和 [Infino](https://github.com/infinohq/infino) 调用 OpenAI 和 ChatOpenAI 模型时追踪以下内容：

* prompt 输入
* 来自 `ChatGPT` 或任何其他 `LangChain` 模型的响应
* 延迟
* 错误
* 消耗的 token 数量

## 初始化

In [1]:
# Install necessary dependencies.
%pip install --upgrade --quiet  infinopy
%pip install --upgrade --quiet  matplotlib
%pip install --upgrade --quiet  tiktoken
%pip install --upgrade --quiet  langchain langchain-openai langchain-community
%pip install --upgrade --quiet  beautifulsoup4

In [ ]:
from langchain_community.callbacks.infino_callback import InfinoCallbackHandler

In [2]:
import datetime as dt
import json
import time

import matplotlib.dates as md
import matplotlib.pyplot as plt
from infinopy import InfinoClient
from langchain_openai import OpenAI

## 启动 Infino 服务器，初始化 Infino 客户端

In [3]:
# Start server using the Infino docker image.
!docker run --rm --detach --name infino-example -p 3000:3000 infinohq/infino:latest

# Create Infino client.
client = InfinoClient()

a1159e99c6bdb3101139157acee6aba7ae9319375e77ab6fbc79beff75abeca3


## 读取问题数据集

In [4]:
# These are a subset of questions from Stanford's QA dataset -
# https://rajpurkar.github.io/SQuAD-explorer/
data = """In what country is Normandy located?
When were the Normans in Normandy?
From which countries did the Norse originate?
Who was the Norse leader?
What century did the Normans first gain their separate identity?
Who gave their name to Normandy in the 1000's and 1100's
What is France a region of?
Who did King Charles III swear fealty to?
When did the Frankish identity emerge?
Who was the duke in the battle of Hastings?
Who ruled the duchy of Normandy
What religion were the Normans
What type of major impact did the Norman dynasty have on modern Europe?
Who was famed for their Christian spirit?
Who assimilted the Roman language?
Who ruled the country of Normandy?
What principality did William the conqueror found?
What is the original meaning of the word Norman?
When was the Latin version of the word Norman first recorded?
What name comes from the English words Normans/Normanz?"""

questions = data.split("\n")

## 示例 1：LangChain OpenAI 问答；将指标和日志发布到 Infino

This example demonstrates how to integrate LangChain with OpenAI and publish metrics and logs to Infino.

本示例演示了如何将 LangChain 与 OpenAI 集成，并将指标和日志发布到 Infino。

**Prerequisites**

**先决条件**

*   LangChain installed
*   OpenAI API key
*   Infino account and API key

*   已安装 LangChain
*   OpenAI API 密钥
*   Infino 账户和 API 密钥

**Steps**

**步骤**

1.  **Set up your environment:**
    *   Install necessary libraries:
        ```bash
        pip install langchain openai infino
        ```
    *   Set your OpenAI API key as an environment variable:
        ```bash
        export OPENAI_API_KEY='your-openai-api-key'
        ```
    *   Set your Infino API key and endpoint as environment variables:
        ```bash
        export INFINO_API_KEY='your-infino-api-key'
        export INFINO_ENDPOINT='your-infino-endpoint'
        ```

2.  **Set up your environment:**
    *   Install necessary libraries:
        ```bash
        pip install langchain openai infino
        ```
    *   Set your OpenAI API key as an environment variable:
        ```bash
        export OPENAI_API_KEY='your-openai-api-key'
        ```
    *   Set your Infino API key and endpoint as environment variables:
        ```bash
        export INFINO_API_KEY='your-infino-api-key'
        export INFINO_ENDPOINT='your-infino-endpoint'
        ```

3.  **Create a LangChain application:**

    ```python
    from langchain_openai import ChatOpenAI
    from langchain.chains import LLMChain
    from langchain.prompts import PromptTemplate
    from infino import InfinoCallbackHandler

    # Initialize the LLM
    llm = ChatOpenAI(model_name="gpt-3.5-turbo")

    # Define a prompt template
    template = """Question: {question}

    Answer: """
    prompt = PromptTemplate(template=template, input_variables=["question"])

    # Create the LLM chain
    llm_chain = LLMChain(prompt=prompt, llm=llm)

    # Initialize Infino callback handler
    infino_callback = InfinoCallbackHandler()

    # Ask a question and run the chain with Infino callbacks
    question = "What is the capital of France?"
    response = llm_chain.invoke({"question": question}, config={"callbacks": [infino_callback]})

    print(response)
    ```

4.  **Run the application:**
    *   Save the code as a Python file (e.g., `app.py`).
    *   Run the file from your terminal:
        ```bash
        python app.py
        ```

This will execute the LangChain application, and the metrics and logs will be automatically published to your Infino account. You can then view them in the Infino dashboard.

这将执行 LangChain 应用程序，并且指标和日志将自动发布到您的 Infino 账户。然后您可以在 Infino 仪表板中查看它们。

In [5]:
# Set your key here.
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# Create callback handler. This logs latency, errors, token usage, prompts as well as prompt responses to Infino.
handler = InfinoCallbackHandler(
    model_id="test_openai", model_version="0.1", verbose=False
)

# Create LLM.
llm = OpenAI(temperature=0.1)

# Number of questions to ask the OpenAI model. We limit to a short number here to save $$ while running this demo.
num_questions = 10

questions = questions[0:num_questions]
for question in questions:
    print(question)

    # We send the question to OpenAI API, with Infino callback.
    llm_result = llm.generate([question], callbacks=[handler])
    print(llm_result)

In what country is Normandy located?
generations=[[Generation(text='\n\nNormandy is located in France.', generation_info={'finish_reason': 'stop', 'logprobs': None})]] llm_output={'token_usage': {'total_tokens': 16, 'prompt_tokens': 7, 'completion_tokens': 9}, 'model_name': 'text-davinci-003'} run=[RunInfo(run_id=UUID('67a516e3-d48a-4e83-92ba-a139079bd3b1'))]
When were the Normans in Normandy?
generations=[[Generation(text='\n\nThe Normans first settled in Normandy in the late 9th century.', generation_info={'finish_reason': 'stop', 'logprobs': None})]] llm_output={'token_usage': {'total_tokens': 24, 'prompt_tokens': 8, 'completion_tokens': 16}, 'model_name': 'text-davinci-003'} run=[RunInfo(run_id=UUID('6417a773-c863-4942-9607-c8a0c5d486e7'))]
From which countries did the Norse originate?
generations=[[Generation(text='\n\nThe Norse originated from Scandinavia, which includes the modern-day countries of Norway, Sweden, and Denmark.', generation_info={'finish_reason': 'stop', 'logprobs

## 创建指标图表

我们现在使用 matplotlib 来创建延迟、错误和消耗的 token 的图表。

In [6]:
# Helper function to create a graph using matplotlib.
def plot(data, title):
    data = json.loads(data)

    # Extract x and y values from the data
    timestamps = [item["time"] for item in data]
    dates = [dt.datetime.fromtimestamp(ts) for ts in timestamps]
    y = [item["value"] for item in data]

    plt.rcParams["figure.figsize"] = [6, 4]
    plt.subplots_adjust(bottom=0.2)
    plt.xticks(rotation=25)
    ax = plt.gca()
    xfmt = md.DateFormatter("%Y-%m-%d %H:%M:%S")
    ax.xaxis.set_major_formatter(xfmt)

    # Create the plot
    plt.plot(dates, y)

    # Set labels and title
    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.title(title)

    plt.show()

In [ ]:
response = client.search_ts("__name__", "latency", 0, int(time.time()))
plot(response.text, "Latency")

response = client.search_ts("__name__", "error", 0, int(time.time()))
plot(response.text, "Errors")

response = client.search_ts("__name__", "prompt_tokens", 0, int(time.time()))
plot(response.text, "Prompt Tokens")

response = client.search_ts("__name__", "completion_tokens", 0, int(time.time()))
plot(response.text, "Completion Tokens")

response = client.search_ts("__name__", "total_tokens", 0, int(time.time()))
plot(response.text, "Total Tokens")

## 对提示或提示输出进行全文搜索。

In [8]:
# Search for a particular prompt text.
query = "normandy"
response = client.search_log(query, 0, int(time.time()))
print("Results for", query, ":", response.text)

print("===")

query = "king charles III"
response = client.search_log("king charles III", 0, int(time.time()))
print("Results for", query, ":", response.text)

Results for normandy : [{"time":1696947743,"fields":{"prompt_response":"\n\nThe Normans, a people from northern France, gave their name to Normandy in the 1000s and 1100s. The Normans were descendants of Vikings who had settled in the region in the late 800s."},"text":"\n\nThe Normans, a people from northern France, gave their name to Normandy in the 1000s and 1100s. The Normans were descendants of Vikings who had settled in the region in the late 800s."},{"time":1696947740,"fields":{"prompt":"Who gave their name to Normandy in the 1000's and 1100's"},"text":"Who gave their name to Normandy in the 1000's and 1100's"},{"time":1696947733,"fields":{"prompt_response":"\n\nThe Normans first settled in Normandy in the late 9th century."},"text":"\n\nThe Normans first settled in Normandy in the late 9th century."},{"time":1696947732,"fields":{"prompt_response":"\n\nNormandy is located in France."},"text":"\n\nNormandy is located in France."},{"time":1696947731,"fields":{"prompt":"In what coun

# 示例 2：使用 ChatOpenAI 总结一段文本

In [9]:
# Set your key here.
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

from langchain.chains.summarize import load_summarize_chain
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI

# Create callback handler. This logs latency, errors, token usage, prompts, as well as prompt responses to Infino.
handler = InfinoCallbackHandler(
    model_id="test_chatopenai", model_version="0.1", verbose=False
)

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://medium.com/lyft-engineering/lyftlearn-ml-model-training-infrastructure-built-on-kubernetes-aef8218842bb",
    "https://blog.langchain.dev/week-of-10-2-langchain-release-notes/",
]

for url in urls:
    loader = WebBaseLoader(url)
    docs = loader.load()

    llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo-16k", callbacks=[handler])
    chain = load_summarize_chain(llm, chain_type="stuff", verbose=False)

    chain.run(docs)

## 创建指标图表

This guide explains how to create metric charts in your application.

本指南将介绍如何在您的应用程序中创建指标图表。

**Prerequisites**

**先决条件**

*   A basic understanding of how to use your application.
*   Access to the application's metric data.

*   对如何使用您的应用程序有基本了解。
*   可以访问应用程序的指标数据。

**Steps**

**步骤**

1.  **Navigate to the Charts section.**
    *   Click on the **Charts** tab in the main navigation menu.

2.  **Click the "Create New Chart" button.**
    *   This button is usually located at the top right of the Charts page.

3.  **Select a chart type.**
    *   A modal will appear, allowing you to choose from various chart types (e.g., line chart, bar chart, pie chart).
    *   Select the chart type that best represents your metric data.

4.  **Configure your chart.**
    *   **Data Source:** Choose the metric data you want to visualize. You may need to select a specific data source or API endpoint.
    *   **X-axis:** Define what will be displayed on the horizontal axis. This is often a time series or a category.
    *   **Y-axis:** Define what will be displayed on the vertical axis. This is typically the value of your metric.
    *   **Filters (Optional):** Apply filters to narrow down the data displayed in the chart. For example, you might filter by date range, user segment, or region.
    *   **Chart Title:** Give your chart a descriptive title.
    *   **Labels and Tooltips:** Customize axis labels and tooltips for better readability.

5.  **Preview your chart.**
    *   Most applications provide a preview option to see how your chart will look before saving.

6.  **Save your chart.**
    *   Click the **Save** button. You may be prompted to name your chart and choose a location or dashboard to save it to.

**Example: Creating a Line Chart for User Signups**

**示例：为用户注册创建折线图**

Let's say you want to track daily user signups.

假设您想跟踪每日用户注册量。

1.  Go to the **Charts** section.
2.  Click **Create New Chart**.
3.  Select **Line Chart**.
4.  Configure the chart:
    *   **Data Source:** `User Signups`
    *   **X-axis:** `Date` (daily aggregation)
    *   **Y-axis:** `Count of Signups`
    *   **Chart Title:** `Daily User Signups`
    *   **Filters:** Set the date range to the last 30 days.
5.  Preview the chart to ensure it looks correct.
6.  Click **Save** and name it `Daily Signups Trend`.

1.  转到 **图表** 部分。
2.  单击 **创建新图表**。
3.  选择 **折线图**。
4.  配置图表：
    *   **数据源**：`用户注册`
    *   **X轴**：`日期`（每日聚合）
    *   **Y轴**：`注册数量`
    *   **图表标题**：`每日用户注册量`
    *   **筛选器**：将日期范围设置为过去 30 天。
5.  预览图表以确保其外观正确。
6.  单击 **保存** 并命名为 `每日注册趋势`。

**Tips for Effective Charts**

**有效图表的技巧**

*   **Choose the right chart type:** Use line charts for trends over time, bar charts for comparisons, and pie charts for proportions.
*   **Keep it simple:** Avoid cluttering your chart with too much information.
*   **Use clear labels:** Ensure your axes and data points are clearly labeled.
*   **Consider your audience:** Tailor the complexity and detail of your charts to who will be viewing them.

*   **选择正确的图表类型**：使用折线图表示随时间变化的趋势，使用条形图进行比较，使用饼图表示比例。
*   **保持简洁**：避免在图表中填充过多信息而显得杂乱。
*   **使用清晰的标签**：确保您的坐标轴和数据点都有清晰的标签。
*   **考虑您的受众**：根据查看图表的人员，调整图表的复杂性和详细程度。

In [ ]:
response = client.search_ts("__name__", "latency", 0, int(time.time()))
plot(response.text, "Latency")

response = client.search_ts("__name__", "error", 0, int(time.time()))
plot(response.text, "Errors")

response = client.search_ts("__name__", "prompt_tokens", 0, int(time.time()))
plot(response.text, "Prompt Tokens")

response = client.search_ts("__name__", "completion_tokens", 0, int(time.time()))
plot(response.text, "Completion Tokens")

In [11]:
## Full text query on prompt or prompt outputs

In [12]:
# Search for a particular prompt text.
query = "machine learning"
response = client.search_log(query, 0, int(time.time()))

# The output can be verbose - uncomment below if it needs to be printed.
# print("Results for", query, ":", response.text)

print("===")

===


In [13]:
## Stop Infino server

In [14]:
!docker rm -f infino-example

infino-example
